# Group Homework 3 · The GoMart report makeover
**Data Visualization · UEH · Session 3** · team notebook

| | |
|---|---|
| **Team** | *team name* · **team number** (from the LMS list): __ |
| **Status** | Formative: written feedback with the rubric in the brief. Task 0 is the first version of Module M3 of your project |
| **Submit** | File ▸ Download ▸ .ipynb as `GH3_TeamName.ipynb` + the evidence folder of Task 4, within 7 days on the LMS |

Read the brief *Group Homework 3* first. All GoMart data is simulated.

## Task 5 · One-page summary for the operations director (write this last)
**Big Idea:** *one complete sentence.*

*At most 100 words. The one-page figure is made in the last code cell of the notebook (it needs everything above);
after "Restart and run all", paste its image here.*

## Task 0 · Project: house style v0 and makeover plan (Module M3)
**Allocated project:** P__ · *title*

1. Put your `house_style.py` v0 in the cell below (palette in fixed order, sequential map, diverging map, fonts, number format).
2. Show the evidence: a swatch of your palette in colour, in greyscale and with simulated deuteranopia.
3. Makeover plan for one published chart from your domain: link, screenshot, critique table, photo of your hand sketch.

In [1]:
# Task 0 · house_style.py v0 for your project (paste the file content here, then run it)
# PROJECT_PALETTE = [...]       # fixed order, one colour per category of your project
# SEQ_CMAP = "..."; DIV_CMAP = "..."
# def use_house_style(): ...

In [2]:
# Task 0 · evidence: palette swatch in colour, greyscale, deuteranopia
# (use fig_to_rgb, to_grey and simulate_cvd from the Setup section below)

**Makeover plan.** Source (link, date accessed): ...

| # | What I see | Principle broken (name it) | Effect on the reader | My fix |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |
| 4 | | | | |

*Insert the photo of your hand sketch here.*

## Setup
Upload `gomart_orders_2025_clean.csv`, `gomart_districts.csv` and `gomart_targets_2025.csv`.
Run `gomart_bad_report.py` once (or open `gomart_bad_report.pdf`) to see the six charts you will audit.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import TwoSlopeNorm, to_rgb

GREY, LIGHT, DARK, BLUE, ORANGE = "#a3a29c", "#d9d8d3", "#52514e", "#2a78d6", "#eb6834"
ZONE_COLOR = {"Central": "#a9c9ef", "Inner": "#5b9be3", "Outer": "#1f4f8f"}   # ordered: light -> dark

In [3]:
from pathlib import Path

# Find the repository's data folder whether the kernel starts at the repo root
# or in this notebook's nested workspace folder.
search_roots = [Path.cwd(), *Path.cwd().parents]
data_dir = next((root / "data" / "raw" for root in search_roots
                 if (root / "data" / "raw" / "gomart_orders_2025_clean.csv").exists()), None)
if data_dir is None:
    raise FileNotFoundError("Could not find data/raw/gomart_orders_2025_clean.csv. Start the notebook from the repository or place the GoMart CSV files in data/raw/.")

orders = pd.read_csv(data_dir / "gomart_orders_2025_clean.csv",
                     parse_dates=["order_time"])
districts = pd.read_csv(data_dir / "gomart_districts.csv")
targets = pd.read_csv(data_dir / "gomart_targets_2025.csv")
orders = orders.merge(districts, on="district", how="left", validate="many_to_one")
orders["month"] = orders["order_time"].dt.strftime("%Y-%m")
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
orders["weekday"] = pd.Categorical(orders["order_time"].dt.strftime("%a"), categories=days, ordered=True)
orders["hour"] = orders["order_time"].dt.hour
tiers = ["none", "Silver", "Gold", "Platinum"]
orders["member_tier"] = pd.Categorical(orders["member_tier"], categories=tiers, ordered=True)
delivered = orders[orders["status"] == "delivered"]
print(orders.shape, "| orders without zone:", orders["zone"].isna().sum())

FileNotFoundError: [Errno 2] No such file or directory: 'c:\\HW3\\DataViz_Day03_Group\\DataViz_Day03_Group\\homework\\gh3_gomart\\data\\raw\\data\\group_homework\\gomart_orders_2025_clean.csv'

Helper functions from Tutorial 3 (given):

In [8]:
def fig_to_rgb(fig):
    fig.canvas.draw()
    return np.asarray(fig.canvas.buffer_rgba())[..., :3] / 255

def to_grey(rgb):
    return np.asarray(rgb)[..., :3] @ np.array([0.299, 0.587, 0.114])

DEUTAN = np.array([[0.367322, 0.860646, -0.227968], [0.280085, 0.672501, 0.047413],
                   [-0.011820, 0.042940, 0.968881]])

def simulate_cvd(rgb, M=DEUTAN):
    lin = np.where(rgb <= 0.04045, rgb / 12.92, ((rgb + 0.055) / 1.055) ** 2.4)
    sim = np.clip(lin @ M.T, 0, 1)
    return np.where(sim <= 0.0031308, 12.92 * sim, 1.055 * sim ** (1 / 2.4) - 0.055)

## Task 1 ? Audit the report
For each of the six charts: the task the chart should support (action + target), then at least four problems.

| Chart | Task it should support | Problem | Principle (name it) | Effect on the reader |
|---|---|---|---|---|
| 1 Orders by district | Compare order volume by district to locate the highest-demand areas | Pie slices are all fairly similar in size | Use position on a common scale for precise comparison | Hard to rank districts or see small differences |
| 1 | Same | Ten saturated rainbow colours have no meaning | Use a restrained, consistent palette | Colours compete for attention and imply categories that do not matter |
| 1 | Same | Exploded slices and shadow separate and distort the whole | Preserve visual integrity | The pie no longer reads as one whole and slice sizes are harder to compare |
| 1 | Same | Outside labels collide with the pie and are repeated in a legend | Direct labeling and avoid redundant encoding | Readers must scan back and forth to match names, colours, and percentages |
| 1 | Same | Only percentages are labeled | Show useful context and units | Readers cannot tell the actual order counts or how large the dataset is |
| 2 Payments | Track monthly order volume and the share of orders paid by e-wallet | Two unrelated measures share a dual-axis plot | Avoid dual axes; use aligned panels | Similar-looking slopes can suggest a relationship that the scales do not establish |
| 2 | Same | Both y-axes are truncated (orders 380?560; share 25?65%) | Use honest baselines | Changes look much larger than they are |
| 2 | Same | Every month uses a different rainbow colour | Use colour consistently | Month colours add no data and make the bars harder to compare |
| 2 | Same | Heavy black grid and two separate legends add clutter | Reduce non-data ink and label directly | The reader must spend effort decoding the chart |
| 2 | Same | Month labels are rotated and dense | Make labels easy to scan | Month-to-month reading is slower |
| ... | | | | |

### Chart 1 ? Orders by district
*Task:* Compare order volume across districts to identify where demand is highest. *Why this chart:* A sorted horizontal bar chart makes small differences easier to compare than pie slices; direct labels show both order count and share.

In [ ]:
# Orders by district: sorted bars support accurate comparisons.
district_counts = orders.groupby("district").size().sort_values()
district_share = district_counts / district_counts.sum() * 100
fig, ax = plt.subplots(figsize=(9, 5.5))
bars = ax.barh(district_counts.index, district_counts.values, color=BLUE, height=0.72)
ax.bar_label(bars, labels=[f"{n:,}  ({p:.1f}%)" for n, p in zip(district_counts.values, district_share.values)],
             padding=5, fontsize=9)
ax.set_title("Orders are spread across districts, with Thu Duc City leading", loc="left", pad=14, weight="bold")
ax.set_xlabel("Number of orders")
ax.set_ylabel("District")
ax.set_xlim(0, district_counts.max() * 1.22)
ax.xaxis.set_major_locator(mticker.MultipleLocator(100))
ax.grid(axis="x", color=LIGHT, linewidth=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.spines["bottom"].set_color(LIGHT)
fig.tight_layout()
plt.show()

In [ ]:
# TODO: write your code here

### Chart 2 ? Payments
*Task:* Track monthly order volume and the share of orders paid by e-wallet. *Why this chart:* Two vertically aligned panels keep each measure on its own honest scale while preserving month-by-month alignment.

#### Lời thoại: Các vấn đề của chart 2 cũ

“Chart 2 cũ thể hiện số lượng đơn hàng và tỷ lệ thanh toán bằng ví điện tử theo tháng. Tuy nhiên, biểu đồ dùng hai trục tung với thang đo khác nhau, nên người xem có thể hiểu nhầm mức độ thay đổi hoặc cho rằng hai đại lượng có quan hệ trực tiếp. Cả hai trục đều bị cắt gốc: số đơn bắt đầu từ 380 thay vì 0, còn tỷ lệ ví điện tử bắt đầu từ 25%, khiến biến động trông lớn hơn thực tế. Mỗi tháng còn có một màu cầu vồng khác nhau dù màu không biểu thị thêm thông tin. Lưới đậm, hai legend và nhãn tháng xoay khiến biểu đồ thêm rối.”

In [ ]:
# Standalone setup: load the source data directly so this cell does not depend on prior cells.
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

search_roots = [Path.cwd(), *Path.cwd().parents]
data_dir = next((root / "data" / "raw" for root in search_roots
                 if (root / "data" / "raw" / "gomart_orders_2025_clean.csv").exists()), None)
if data_dir is None:
    raise FileNotFoundError(
        "Cannot find data/raw/gomart_orders_2025_clean.csv. "
        "Run this notebook from the repository or put the GoMart CSV files in data/raw/."
    )
orders = pd.read_csv(data_dir / "gomart_orders_2025_clean.csv", parse_dates=["order_time"])
orders["month"] = orders["order_time"].dt.strftime("%Y-%m")

PAYMENT_BLUE = "#2a78d6"
PAYMENT_ORANGE = "#eb6834"
PAYMENT_LIGHT = "#d9d8d3"

# Monthly order volume and e-wallet share, with separate y-scales.
month_order = [f"2025-{m:02d}" for m in range(1, 13)]
monthly = (orders.groupby("month")
           .agg(order_count=("order_id", "size"),
                ewallet_share=("payment", lambda s: (s == "E-wallet").mean() * 100))
           .reindex(month_order))
month_labels = pd.to_datetime(month_order).strftime("%b")
x = np.arange(len(month_order))
fig, (ax_orders, ax_share) = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True,
                                            gridspec_kw={"height_ratios": [1, 1.15]})
bars = ax_orders.bar(x, monthly["order_count"], color=PAYMENT_BLUE, width=0.68)
ax_orders.bar_label(bars, labels=[f"{int(v)}" for v in monthly["order_count"]], padding=3, fontsize=8)
ax_orders.set_ylim(0, 600)
ax_orders.set_ylabel("Orders")
ax_orders.yaxis.set_major_locator(mticker.MultipleLocator(100))
ax_orders.grid(axis="y", color=PAYMENT_LIGHT, linewidth=0.8)
ax_orders.set_axisbelow(True)
ax_share.plot(x, monthly["ewallet_share"], color=PAYMENT_ORANGE, marker="o",
               linewidth=2.5, markersize=6)
ax_share.set_ylim(0, 100)
ax_share.set_yticks(np.arange(0, 101, 20), labels=[f"{v}%" for v in range(0, 101, 20)])
ax_share.set_ylabel("E-wallet share")
ax_share.set_xticks(x, month_labels)
ax_share.set_xlabel("Month in 2025")
ax_share.grid(axis="y", color=PAYMENT_LIGHT, linewidth=0.8)
ax_share.set_axisbelow(True)
ax_share.annotate(f"{monthly.ewallet_share.iloc[0]:.0f}%", (x[0], monthly.ewallet_share.iloc[0]),
                  xytext=(0, 10), textcoords="offset points", ha="center", color=PAYMENT_ORANGE, weight="bold")
ax_share.annotate(f"{monthly.ewallet_share.iloc[-1]:.0f}%", (x[-1], monthly.ewallet_share.iloc[-1]),
                  xytext=(0, 10), textcoords="offset points", ha="center", color=PAYMENT_ORANGE, weight="bold")
for ax in (ax_orders, ax_share):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines["left"].set_color(PAYMENT_LIGHT)
    ax.spines["bottom"].set_color(PAYMENT_LIGHT)
fig.suptitle("E-wallet share rose from 31% to 60% during 2025",
             x=0.08, ha="left", fontsize=14, weight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.95))
plt.show()

#### Lời thoại: Giải thích chart 2 mới

“Biểu đồ mới tách hai chỉ số thành hai phần nhưng giữ chung trục tháng để dễ đối chiếu. Phần trên dùng cột để thể hiện số đơn hàng; phần dưới dùng đường để thể hiện tỷ lệ đơn thanh toán bằng ví điện tử. Hai chỉ số có thang đo riêng bắt đầu từ 0, giúp tránh nhầm lẫn do hai trục tung khác nhau. Tỷ lệ ví điện tử tăng từ khoảng 31% vào tháng 1 lên 60% vào tháng 12 năm 2025, trong khi số đơn hàng dao động theo tháng. Nhờ vậy, người xem có thể đọc rõ từng xu hướng mà không bị đánh lạc hướng bởi màu sắc hay trục kép.”

### Chart 3 · Delivery time
*Task:* ...  *Why this chart:* ...

In [ ]:
# TODO: write your code here

In [ ]:
# TODO: write your code here

### Chart 4 · Peak times
*Task:* ...  *Why this chart:* ...

In [ ]:
# TODO: write your code here

### Chart 5 · Targets
*Task:* ...  *Why this chart:* ...

In [ ]:
# TODO: write your code here

In [ ]:
# TODO: write your code here

### Chart 6 · Basket by tier
*Task:* ...  *Why this chart:* ...

In [ ]:
# TODO: write your code here

In [ ]:
# TODO: write your code here

*Check: every number in your six titles, with the output that proves it.*

## Task 3 · Test for every reader
Greyscale and deuteranopia versions of all six redesigns. Then: one change you made because of a test.

In [ ]:
# TODO: write your code here

*The change we made because of a test, and why:*

## Task 4 · Perception experiment
### 4.1 Your stimulus sheet
Use **your team number** from the LMS list. Print the sheet (or put it in a form). Do not show the true values to anyone.

In [ ]:
def make_stimuli(team_number, path="stimulus_sheet.png"):
    """Perception experiment: one sheet with three panels (bars, pie, bubbles).
    Each panel shows five values A-E; the largest is marked with *. Values differ per panel
    and per team, so every team has its own experiment. Returns the true values (keep them secret)."""
    rng = np.random.default_rng(1000 + team_number)
    truth = {}
    fig, axs = plt.subplots(1, 3, figsize=(13, 4.2))
    for ax, kind in zip(axs, ["bars", "pie", "bubbles"]):
        v = np.round(rng.uniform(10, 90, 5))
        v[rng.integers(5)] = 100                                  # the reference value (only one)
        labels = [L + ("*" if val == 100 else "") for L, val in zip("ABCDE", v)]
        truth[kind] = dict(zip("ABCDE", v.astype(int).tolist()))
        if kind == "bars":
            ax.bar(labels, v, color="white", edgecolor="black")
            ax.set_yticks([]); ax.spines[["top", "right", "left"]].set_visible(False)
        elif kind == "pie":
            ax.pie(v, labels=labels, colors=["white"] * 5, wedgeprops=dict(edgecolor="black"),
                   startangle=90, counterclock=False)
        else:
            ax.scatter(range(5), [0] * 5, s=v / 100 * 2500, color="white", edgecolor="black")
            for i, L in enumerate(labels):
                ax.text(i, -0.55, L, ha="center")
            ax.set_ylim(-0.8, 0.8); ax.set_xlim(-0.7, 4.7); ax.axis("off")
        ax.set_title(f"Panel {['bars', 'pie', 'bubbles'].index(kind) + 1}")
    fig.text(0.5, 0.0, "The value marked * is 100. Estimate the other four values in each panel. "
             "Do not measure; answer within 30 seconds per panel.", ha="center", fontsize=10)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    return truth

In [ ]:
# truth = make_stimuli(team_number=__)
# print(truth)
# TODO: write your code here

### 4.2 Protocol (fill in before you collect)
Who (≥ 15 people not in this course), where, when, how you avoid helping them, how you record answers, consent, no names.

### 4.3 Your answers
Save your answers as `experiment_answers.csv` with the columns
`respondent, encoding, item, true, judged, collector` (one row per estimate; `collector` = the student ID of the team
member who collected it). Upload it next to the notebook.

In [ ]:
# answers = pd.read_csv("experiment_answers.csv")
# answers["log_error"] = np.log2(np.abs(answers["judged"] - answers["true"]) + 1 / 8)   # Cleveland & McGill (1984)
# print(answers.groupby("encoding")["log_error"].agg(["mean", "count"]).round(2))

In [ ]:
# Task 4.3: compute log_error for your own answers and summarise it by encoding
# TODO: write your code here

### 4.4 Result chart

In [ ]:
# TODO: write your code here

### 4.5 Interpretation (at most 150 words)
Does your result agree with Cleveland & McGill (1984)? Which item was hardest, and why? Two limitations of your experiment.

## Task 5 · The one-page figure
Three panels, one action title, consistent colours with Tasks 2-3.

In [ ]:
# Task 5: one figure, three panels, one action title
# TODO: write your code here

## Task 6 · Team and AI
| Member | What they did | Answers collected (Task 4) | Share of work |
|---|---|---|---|
| | | | |

In [ ]:
AI_USE = """
Tool(s) used:
What we asked:
What we checked or changed ourselves:
"""
print(AI_USE)